Scripts to clean Colorado Year 1 data

Find the file

In [2]:
from pathlib import Path
import pandas as pd

# Finds the repo root no matter which folder the notebook is saved in
here = Path.cwd()
repo_root = next(p for p in [here, *here.parents] if (p / "data").exists())

path = repo_root / "data" / "colorado_data" / "year_1_wage.csv"
print(path)
print("file found:", path.exists())

/Users/victoriajohnson/Desktop/CS/BreakThroughTech/Analysis/Sage-Road-Solutions-1B-workforce-pell-opportunity-score/data/colorado_data/year_1_wage.csv
file found: True


Load file as plain text

In [3]:
raw = pd.read_csv(path, dtype=str)
raw.columns = [c.strip() for c in raw.columns]

print(raw.shape)
print(raw.columns.tolist())
raw.head()

(3748, 14)
['Entity Id', 'Entity Name', 'CIPCode', 'CIP Title', 'ROI Report Group', 'Degree Level Id', 'Degree', 'Completers', 'Completers with wage count', '%Completers with wage data', 'Annual QTILE1 Earnings ($)', 'Annual Median Earnings ($)', 'Annual QTILE3 Earnings ($)', 'End of workbook']


,Entity Id,Entity Name,CIPCode,CIP Title,ROI Report Group,Degree Level Id,Degree,Completers,Completers with wage count,%Completers with wage data,Annual QTILE1 Earnings ($),Annual Median Earnings ($),Annual QTILE3 Earnings ($),End of workbook
0,-1,ALL,5112,Medicine.(51.12),Health,61,Doctoral Degree,"1,124",174,15%,"34,665","61,051","67,257",NaN
1,0524,Colorado State University - Pueblo,5138,"Registered Nursing, Nursing Administration, Nu...",Health,61,Doctoral Degree,25,23,92%,"62,990","80,066","134,670",NaN
2,0524,Colorado State University - Pueblo,1601,"Linguistic, Comparative, and Related Language ...","Arts, Humanities and Communication",21,Bachelor's Degree,110,64,58%,"30,215","40,408","52,053",NaN
3,0504,Colorado State University,5007,Fine and Studio Arts.(50.07),"Arts, Humanities and Communication",31,Master's Degree,46,16,35%,"28,922","32,350","49,252",NaN
4,-1,ALL,1302,"Bilingual, Multilingual, and Multicultural Edu...",Education,31,Master's Degree,756,662,88%,"55,479","65,523","78,583",NaN


Clean data

In [4]:
df = raw.copy()

# Drop junk columns (like "End of workbook") and junk rows (like "End of rows")
df = df.loc[:, ~df.columns.str.contains("End of|Unnamed", case=False)]
df = df[~df.iloc[:, 0].fillna("").str.contains("End of", case=False)]

# Give the columns clean names (by position)
names = ["ENTITY_ID", "ENTITY_NAME", "CIPCODE", "CIP_TITLE", "ROI_GROUP",
         "DEGREE_LEVEL_ID", "DEGREE", "COMPLETERS", "COMPLETERS_WITH_WAGE",
         "PCT_WITH_WAGE", "EARN_Q1", "EARN_MEDIAN", "EARN_Q3"]
assert df.shape[1] == len(names), f"Expected 13 columns, found {df.shape[1]}: {df.columns.tolist()}"
df.columns = names

# Flag the suppressed values BEFORE turning them into blanks
df["COMPLETERS_SUPPRESSED"] = df["COMPLETERS"].str.contains("Insufficient", case=False, na=False)
df["EARNINGS_SUPPRESSED"] = df["EARN_MEDIAN"].str.contains("Insufficient", case=False, na=False)

# Turn text into real numbers ("Insufficient Data" becomes blank)
num_cols = ["COMPLETERS", "COMPLETERS_WITH_WAGE", "PCT_WITH_WAGE",
            "EARN_Q1", "EARN_MEDIAN", "EARN_Q3"]
for c in num_cols:
    cleaned = (df[c].str.replace(",", "", regex=False)
                    .str.replace("$", "", regex=False)
                    .str.replace("%", "", regex=False))
    df[c] = pd.to_numeric(cleaned, errors="coerce")

# "15%" is now 15, so turn it into 0.15
df["PCT_WITH_WAGE"] = df["PCT_WITH_WAGE"] / 100

df["CIPCODE"] = pd.to_numeric(df["CIPCODE"], errors="coerce")
for c in ["ENTITY_ID", "ENTITY_NAME", "DEGREE"]:
    df[c] = df[c].str.strip()

print("all rows:", len(df))

all rows: 3747


In [5]:
print(df["PCT_WITH_WAGE"].describe())

count    2338.000000
mean        0.505902
std         0.176720
min         0.060000
25%         0.380000
50%         0.490000
75%         0.630000
max         1.000000
Name: PCT_WITH_WAGE, dtype: float64


In [6]:
print(raw[raw.iloc[:, 0].fillna("").str.contains("End of", case=False)])

            Entity Id Entity Name CIPCode CIP Title ROI Report Group  \
3747  End of workbook         NaN     NaN       NaN              NaN   

     Degree Level Id Degree Completers Completers with wage count  \
3747             NaN    NaN        NaN                        NaN   

     %Completers with wage data Annual QTILE1 Earnings ($)  \
3747                        NaN                        NaN   

     Annual Median Earnings ($) Annual QTILE3 Earnings ($) End of workbook  
3747                        NaN                        NaN             NaN  


Keep only what we need, check, and save

In [7]:
# Keep undergraduate certificates only
cert_types = ["Certificate(<1 year)", "Certificate(>1 year)"]
df = df[df["DEGREE"].isin(cert_types)]
print("undergrad certificate rows:", len(df))

# Drop statewide totals (Entity Id of -1)
df = df[df["ENTITY_ID"] != "-1"]
print("after dropping statewide totals:", len(df))

# Set aside whole-school rollups (CIPCODE of -1), keep real programs
rollup = df[df["CIPCODE"] == -1]
df = df[df["CIPCODE"] != -1].copy()
print("school-wide rollup rows set aside:", len(rollup))
print("program rows kept:", len(df))

# Make school codes 4 digits again in case leading zeros were lost
df["ENTITY_ID"] = df["ENTITY_ID"].str.zfill(4)

# Checks
print("\nschools:", df["ENTITY_ID"].nunique())
print("with Year 1 median earnings:", df["EARN_MEDIAN"].notna().sum(),
      f"({df['EARN_MEDIAN'].notna().mean():.0%})")
print("completers suppressed:", df["COMPLETERS_SUPPRESSED"].sum())

# Save
out = repo_root / "data" / "colorado_data" / "year_1_clean.csv"
df.to_csv(out, index=False)
print("saved to", out)
df.head()

undergrad certificate rows: 992
after dropping statewide totals: 794
school-wide rollup rows set aside: 47
program rows kept: 747

schools: 27
with Year 1 median earnings: 347 (46%)
completers suppressed: 270
saved to /Users/victoriajohnson/Desktop/CS/BreakThroughTech/Analysis/Sage-Road-Solutions-1B-workforce-pell-opportunity-score/data/colorado_data/year_1_clean.csv


,ENTITY_ID,ENTITY_NAME,CIPCODE,CIP_TITLE,ROI_GROUP,DEGREE_LEVEL_ID,DEGREE,COMPLETERS,COMPLETERS_WITH_WAGE,PCT_WITH_WAGE,EARN_Q1,EARN_MEDIAN,EARN_Q3,COMPLETERS_SUPPRESSED,EARNINGS_SUPPRESSED
12,0511,Community College of Denver,1513,Drafting/Design Engineering Technologies/Techn...,"Science, Technology, Engineering and Math",02,Certificate(>1 year),NaN,NaN,NaN,NaN,NaN,NaN,True,True
26,0528,Trinidad State College,5135,Somatic Bodywork and Related Therapeutic Servi...,Health,02,Certificate(>1 year),14.0,NaN,NaN,NaN,NaN,NaN,False,True
29,0507,Front Range Community College,106,Applied Horticulture and Horticultural Busines...,"Science, Technology, Engineering and Math",02,Certificate(>1 year),16.0,NaN,NaN,NaN,NaN,NaN,False,True
31,6326,Technical College of the Rockies,4301,Criminal Justice and Corrections.(43.01),Trades,02,Certificate(>1 year),16.0,14.0,0.88,54887.0,59304.0,67291.0,False,False
38,0497,Arapahoe Community College,5139,"Practical Nursing, Vocational Nursing and Nurs...",Health,01,Certificate(<1 year),1159.0,387.0,0.33,28793.0,36988.0,47095.0,False,False


In [8]:
print((repo_root / "data" / "colorado_data" / "year_1_clean.csv").exists())

True
